# TKAN v4 — Training Diagnostic

Trains **cycle_000** (all data before 2015-01-02) from scratch with visible epoch-by-epoch val_loss,
then runs OOS predictions for the full 2015 year and shows the interactive chart with the 10-day
**input window** highlighted on every forecast step.

In [1]:
import os, sys, pickle, json, warnings, time
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import numpy as np
import pandas as pd
import tensorflow as tf
from tkan import TKAN
from sklearn.preprocessing import RobustScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    _HERE = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _HERE = os.path.abspath('')

_ROOT     = os.path.normpath(os.path.join(_HERE, *(['..'] * 7)))
_SFERA_DB = os.path.join(_ROOT, 'sfera-db')
if os.path.isdir(_SFERA_DB) and _SFERA_DB not in sys.path:
    sys.path.insert(0, _SFERA_DB)
import sfera_db

WEIGHTS_DIR     = os.path.join(_HERE, 'weights')

# ── Config ────────────────────────────────────────────────────────────────
WINDOW_SIZE     = 10
PREDICTION_DAYS = 10
EPOCHS          = 80
BATCH_SIZE      = 32
DROPOUT_RATE    = 0.2
IVOL_WINDOW     = 63      # matches v2 (was 126)
BACKTEST_START  = '2015-01-02'
RETRAIN_FREQ    = 252
ENTRY_THRESHOLD = 1.015
TARGET_COLS     = [f'd{k}' for k in range(1, PREDICTION_DAYS + 1)]

# 2.0 = simulate 2× leveraged ETF daily moves (matches LVC in v2)
# 1.0 = plain CACT total return (no leverage)
LEVERAGE_FACTOR = 2.0

FEATURE_COLS = [
    'log_return_1d', 'high_low_range', 'close_to_high',
    'close_vs_sma15',
    'ivol_zscore', 'ivol_ema_ratio', 'ivol_pctl', 'ivol_roc5',
    'rvol_park20_zscore', 'vol_spread',
    'return_5d', 'return_20d',
]

print(f'TF {tf.__version__}  |  features: {len(FEATURE_COLS)}  |  window: {WINDOW_SIZE}  pred: {PREDICTION_DAYS}')
print(f'EPOCHS={EPOCHS}  BATCH={BATCH_SIZE}  DROPOUT={DROPOUT_RATE}  IVOL_WINDOW={IVOL_WINDOW}')
print(f'LEVERAGE_FACTOR={LEVERAGE_FACTOR}×  (daily log returns scaled by {LEVERAGE_FACTOR})')


TF 2.20.0  |  features: 12  |  window: 10  pred: 10
EPOCHS=80  BATCH=32  DROPOUT=0.2  IVOL_WINDOW=63
LEVERAGE_FACTOR=2.0×  (daily log returns scaled by 2.0)


## 1. Load Data

In [2]:
def _q(sql):
    return (sfera_db.query(sql)
            .assign(date=lambda d: pd.to_datetime(d['date']))
            .set_index('date'))

cactr    = _q("SELECT trade_date AS date, close_price AS close "
              "FROM bbgidx.index_total_return WHERE ticker='CACT' ORDER BY trade_date")
cac_ohlc = _q("SELECT trade_date AS date, open_price AS open, "
              "high_price AS high, low_price AS low, close_price AS cac_close "
              "FROM bbgidx.index_prices WHERE ticker='CAC' ORDER BY trade_date")
ivol_raw = _q("SELECT trade_date AS date, \"3m_50d_ivol\" AS ivol "
              "FROM bbgidx.index_implied_vol WHERE ticker='CAC' ORDER BY trade_date")[['ivol']]

print(f'CACT TR  : {cactr.index[0].date()} → {cactr.index[-1].date()}  ({len(cactr):,} rows)')
print(f'CAC OHLC : {cac_ohlc.index[0].date()} → {cac_ohlc.index[-1].date()}  ({len(cac_ohlc):,} rows)')
print(f'CAC IVol : {ivol_raw.index[0].date()} → {ivol_raw.index[-1].date()}  ({len(ivol_raw):,} rows)')
print()
print('IVol starts 2007-01-02 → effective training data from ~2007 (126-bar warmup = mid-2007)')

CACT TR  : 2000-01-03 → 2026-04-17  (6,725 rows)
CAC OHLC : 2000-01-03 → 2026-04-17  (6,722 rows)
CAC IVol : 2007-01-02 → 2026-04-17  (4,936 rows)

IVol starts 2007-01-02 → effective training data from ~2007 (126-bar warmup = mid-2007)


## 2. Build Features

In [3]:
# Forward-fill ivol over any gaps
ivol_filled = ivol_raw.reindex(ivol_raw.index.union(cactr.index)).ffill()
common = cactr.index.intersection(cac_ohlc.index).intersection(ivol_filled.index)

df = cac_ohlc.loc[common].copy()
df['close'] = cactr.loc[common, 'close']
df['ivol']  = ivol_filled.loc[common, 'ivol']

# ── Synthetic leveraged close ──────────────────────────────────────────────
# LEVERAGE_FACTOR=2 → simulates 2× ETF daily moves (same as LVC in v2)
# LEVERAGE_FACTOR=1 → plain CACT (no scaling)
log_ret_raw   = np.log(df['close'] / df['close'].shift(1)).fillna(0)
df['close_lev'] = df['close'].iloc[0] * np.exp((LEVERAGE_FACTOR * log_ret_raw).cumsum())

# ── Log-based features (all scale-invariant — matching v2 formulas) ────────
# Returns: computed on leveraged series so they represent actual trading moves
df['log_return_1d'] = np.log(df['close_lev'] / df['close_lev'].shift(1))
df['return_5d']     = np.log(df['close_lev'] / df['close_lev'].shift(5))
df['return_20d']    = np.log(df['close_lev'] / df['close_lev'].shift(20))

# Range / position features: from CAC OHLC (CACT has no intraday data)
# Scaled by LEVERAGE_FACTOR to match the leveraged series magnitude
df['high_low_range'] = LEVERAGE_FACTOR * np.log(df['high'] / df['low'].replace(0, np.nan))
df['close_to_high']  = LEVERAGE_FACTOR * np.log(df['cac_close'] / df['high'].replace(0, np.nan))

# SMA distance: log ratio on leveraged series (matches v2: log(close/sma15))
sma15                = df['close_lev'].rolling(15, min_periods=1).mean()
df['close_vs_sma15'] = np.log(df['close_lev'] / sma15)

# ── Vol features ───────────────────────────────────────────────────────────
# Parkinson realized vol from CAC OHLC, annualised in % units (× 100 matches v2)
log_hl        = np.log(df['high'] / df['low'].replace(0, np.nan))
rvol_park20   = np.sqrt((1 / (4 * np.log(2))) * (log_hl ** 2).rolling(20).mean() * 252) * 100
df['rvol_park20'] = rvol_park20

ivol   = df['ivol']
ewma20 = ivol.ewm(span=20).mean()

df['ivol_zscore']        = (ivol - ivol.rolling(IVOL_WINDOW).mean()) / (ivol.rolling(IVOL_WINDOW).std() + 1e-9)
df['ivol_ema_ratio']     = ivol / (ewma20 + 1e-9)
df['ivol_pctl']          = ivol.rolling(IVOL_WINDOW, min_periods=20).rank(pct=True)   # matches v2
df['ivol_roc5']          = ivol.pct_change(5)
df['rvol_park20_zscore'] = (rvol_park20 - rvol_park20.rolling(IVOL_WINDOW).mean()) / \
                            (rvol_park20.rolling(IVOL_WINDOW).std() + 1e-9)
df['vol_spread']         = ivol - rvol_park20   # both in % units

# ── Lag all features by 1 day — no same-day lookahead ─────────────────────
for col in FEATURE_COLS:
    df[col] = df[col].shift(1)

df = df.dropna(subset=FEATURE_COLS + ['close', 'close_lev'])

print(f'Full dataset: {len(df):,} rows  {df.index[0].date()} → {df.index[-1].date()}')
print(f'CACT daily std: {log_ret_raw.std()*100:.3f}%  →  {LEVERAGE_FACTOR}× leveraged: {log_ret_raw.std()*LEVERAGE_FACTOR*100:.3f}%')
print()
print('Feature stats (full history):')
print(df[FEATURE_COLS].describe().round(4).to_string())


Full dataset: 4,853 rows  2007-03-30 → 2026-03-24
CACT daily std: 1.370%  →  2.0× leveraged: 2.740%

Feature stats (full history):
       log_return_1d  high_low_range  close_to_high  close_vs_sma15  ivol_zscore  ivol_ema_ratio  ivol_pctl  ivol_roc5  rvol_park20_zscore  vol_spread  return_5d  return_20d
count      4853.0000       4853.0000      4853.0000       4853.0000    4853.0000       4853.0000  4853.0000  4853.0000           4853.0000   4853.0000  4853.0000   4853.0000
mean          0.0004          0.0287        -0.0139          0.0021      -0.0779          0.9999     0.4665     0.0047             -0.0252      4.8291     0.0020      0.0085
std           0.0270          0.0193         0.0157          0.0542       1.2757          0.0831     0.3280     0.0982              1.3622      3.4199     0.0580      0.1098
min          -0.2620          0.0028        -0.1705         -0.5297      -2.8213          0.7771     0.0159    -0.3206             -3.8624    -19.1459    -0.5637     -0.9740

## 3. Training Window for Cycle 000

**cycle_000** retrains at **2015-01-02**. Training data = everything strictly before that date.

In [4]:
retrain_date = pd.Timestamp(BACKTEST_START)
all_dates    = df.index

# Training: all rows before retrain_date
train_df = df[df.index < retrain_date]
X_train  = train_df[FEATURE_COLS].values
y_close  = train_df['close_lev'].values   # leveraged price series → targets have 2× variance

# OOS: retrain_date ... retrain_date + RETRAIN_FREQ (= full 2015)
oos_start_idx = all_dates.searchsorted(retrain_date)
oos_end_idx   = oos_start_idx + RETRAIN_FREQ
oos_dates     = all_dates[oos_start_idx : oos_end_idx]

print(f'Training window : {train_df.index[0].date()} → {train_df.index[-1].date()}  ({len(train_df):,} rows)')
print(f'OOS window      : {oos_dates[0].date()} → {oos_dates[-1].date()}  ({len(oos_dates):,} days)')
print(f'LEVERAGE_FACTOR : {LEVERAGE_FACTOR}×')
print()

# Scale X (RobustScaler on train only)
scaler = RobustScaler()
X_sc   = scaler.fit_transform(X_train)

# Build sequences: target[k] = close_lev[anchor+k+1] / close_lev[anchor]
# With LEVERAGE_FACTOR=2, target std is ~2× larger than plain CACT
# → MSE gradient is ~4× stronger → model learns instead of collapsing
Xs, ys = [], []
for i in range(len(X_sc) - WINDOW_SIZE - PREDICTION_DAYS):
    Xs.append(X_sc[i : i + WINDOW_SIZE])
    anchor_idx = i + WINDOW_SIZE - 1
    anchor     = y_close[anchor_idx]
    fwd_prices = y_close[anchor_idx + 1 : anchor_idx + PREDICTION_DAYS + 1]
    ys.append(fwd_prices / anchor)
Xs = np.array(Xs, dtype='float32')  # (N, 10, 12)
ys = np.array(ys, dtype='float32')  # (N, 10) — price ratios

print(f'Sequences: X={Xs.shape}  y={ys.shape}')
print(f'Target stats:  mean={ys.mean():.5f}  std={ys.std():.5f}  min={ys.min():.4f}  max={ys.max():.4f}')
print(f'  mean-1 = {(ys.mean()-1)*100:+.3f}%   std = {ys.std()*100:.3f}%')
print(f'  (v2/LVC comparable std was ~3-4% — target signal is now strong enough for MSE to learn)')


Training window : 2007-03-30 → 2014-12-31  (1,980 rows)
OOS window      : 2015-01-02 → 2015-12-24  (252 days)
LEVERAGE_FACTOR : 2.0×

Sequences: X=(1960, 10, 12)  y=(1960, 10)
Target stats:  mean=1.00194  std=0.06525  min=0.5824  max=1.4480
  mean-1 = +0.194%   std = 6.525%
  (v2/LVC comparable std was ~3-4% — target signal is now strong enough for MSE to learn)


## 4. Train cycle_000 — Live Epoch / Val Loss

In [5]:
def build_model():
    # Architecture identical to v2:
    # 3×TKAN(return_sequences=True, use_bias=True) + Dense(1) → (batch, 10, 1)
    # Loss: MSE with MAE as metric — exactly as v2 (loss='mse', metrics=['mae'])
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(shape=(WINDOW_SIZE, len(FEATURE_COLS))),
        TKAN(100, return_sequences=True, use_bias=True),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        TKAN(100, return_sequences=True, use_bias=True),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        TKAN(100, return_sequences=True, use_bias=True),
        tf.keras.layers.Dropout(DROPOUT_RATE),
        tf.keras.layers.Dense(1),   # → (batch, 10, 1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

model = build_model()
model.summary()
print(f'Training on {len(Xs):,} sequences  |  val_split=10%  |  loss=MSE  metrics=[MAE]  (same as v2)')


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tkan (TKAN)                     │ (None, 10, 100)        │        36,676 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tkan_1 (TKAN)                   │ (None, 10, 100)        │       161,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tkan_2 (TKAN)                   │ (None, 10, 100)        │       161,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 10, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10, 1)          │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 360,577 (1.38 MB)

 Trainable params: 358,457 (1.37 MB)

 Non-trainable params: 2,120 (8.28 KB)

Training on 1,960 sequences  |  val_split=10%  |  loss=MSE  metrics=[MAE]  (same as v2)


In [6]:
t0   = time.time()
hist = model.fit(
    Xs, ys,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    validation_split = 0.1,
    verbose          = 1,
)
elapsed = time.time() - t0
train_loss = hist.history['loss']
val_loss   = hist.history['val_loss']
stopped_at = len(train_loss)
best_val   = min(val_loss)
best_ep    = val_loss.index(best_val) + 1

print(f'\nTrained {stopped_at} epochs in {elapsed:.0f}s')
print(f'Best val_loss (MSE) = {best_val:.6f}  at epoch {best_ep}')
print(f'Final train_loss    = {train_loss[-1]:.6f}')


Epoch 1/80
56/56 ━━━━━━━━━━━━━━━━━━━━ 10s 44ms/step - loss: 0.1058 - mae: 0.2430 - val_loss: 0.0124 - val_mae: 0.0813
Epoch 2/80
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.0206 - mae: 0.1140 - val_loss: 0.0030 - val_mae: 0.0440
Epoch 3/80
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.0127 - mae: 0.0883 - val_loss: 0.0034 - val_mae: 0.0479
Epoch 4/80
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - loss: 0.0121 - mae: 0.0865 - val_loss: 0.0022 - val_mae: 0.0355
Epoch 5/80
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.0110 - mae: 0.0825 - val_loss: 0.0020 - val_mae: 0.0343
Epoch 6/80
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - loss: 0.0107 - mae: 0.0810 - val_loss: 0.0020 - val_mae: 0.0336
Epoch 7/80
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - loss: 0.0102 - mae: 0.0787 - val_loss: 0.0023 - val_mae: 0.0355
Epoch 8/80
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 0.0099 - mae: 0.0779 - val_loss: 0.0028 - val_mae: 0.0428
Epoch 9/80
56/56 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.01

## 5. Val Loss Curve

In [7]:
epochs_x = list(range(1, stopped_at + 1))

fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(
    x=epochs_x, y=train_loss, mode='lines+markers', name='train_loss',
    line=dict(color='royalblue', width=2),
    marker=dict(size=4)))
fig_loss.add_trace(go.Scatter(
    x=epochs_x, y=val_loss, mode='lines+markers', name='val_loss',
    line=dict(color='tomato', width=2.5),
    marker=dict(size=5)))

# Mark best val epoch
best_ep = val_loss.index(best_val) + 1
fig_loss.add_vline(x=best_ep, line=dict(color='lime', width=1.5, dash='dot'))
fig_loss.add_annotation(
    x=best_ep, y=best_val, text=f'best={best_val:.6f}<br>ep={best_ep}',
    showarrow=True, arrowhead=2, bgcolor='rgba(0,0,0,0.5)', font=dict(color='lime'))

fig_loss.update_layout(
    title=f'cycle_000  |  train={len(Xs):,} seqs  |  target=price ratios ~1.0  |  MSE',
    xaxis_title='Epoch',
    yaxis_title='MSE Loss',
    template='plotly_dark',
    height=450, width=900,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig_loss.show()

print(f'\nVal loss did {"IMPROVE" if best_ep > 1 else "NOT improve"} — stopped at epoch {stopped_at} (best at {best_ep})')
if stopped_at < EPOCHS:
    print(f'  EarlyStopping fired at epoch {stopped_at}  (patience=8 → best was ep {best_ep})')
else:
    print(f'  WARNING: ran all {EPOCHS} epochs — EarlyStopping never fired, model may still be improving!')


Val loss did IMPROVE — stopped at epoch 80 (best at 12)


## 6. OOS Predictions — cycle_000 → full 2015

In [8]:
full_X   = df[FEATURE_COLS].values
full_sc  = scaler.transform(full_X)    # scale ALL data with cycle_000 scaler
full_lev = df['close_lev'].values      # leveraged prices (for anchoring predictions)

all_preds = {}
for date in oos_dates:
    d_idx = all_dates.get_loc(date)
    if d_idx < WINDOW_SIZE:
        continue
    X_win  = full_sc[d_idx - WINDOW_SIZE : d_idx].reshape(1, WINDOW_SIZE, -1).astype('float32')
    ratios = model(X_win, training=False).numpy()[0, :, 0]  # (10,) price ratios
    all_preds[date] = ratios

pred_df = pd.DataFrame.from_dict(all_preds, orient='index', columns=TARGET_COLS)
pred_df.index = pd.DatetimeIndex(pred_df.index)

print(f'OOS predictions: {len(pred_df):,} rows  {pred_df.index[0].date()} → {pred_df.index[-1].date()}')
print(f'd1  ratio mean={pred_df["d1"].mean():.5f}  std={pred_df["d1"].std():.5f}')
print(f'd10 ratio mean={pred_df["d10"].mean():.5f}  std={pred_df["d10"].std():.5f}')
print(f'd1  pct   mean={(pred_df["d1"].mean()-1)*100:.3f}%  std={pred_df["d1"].std()*100:.3f}%')
print(f'signal ON (max ratio >= {ENTRY_THRESHOLD}): {(pred_df.max(axis=1) >= ENTRY_THRESHOLD).mean():.1%} of days')

close_oos_lev = df.loc[oos_dates, 'close_lev']
real_d1_std   = np.log(close_oos_lev / close_oos_lev.shift(1)).std()
print(f'\nReal {LEVERAGE_FACTOR}× daily log-return std (OOS 2015): {real_d1_std*100:.3f}%')
print(f'Model d1 std (ratio-1):                      {pred_df["d1"].std()*100:.3f}%')
if pred_df['d1'].std() < 1e-4:
    print('  ⚠ Model STILL degenerate — predictions are constant')
else:
    print('  ✓ Predictions vary across days — model is working!')

print(f'\nSample d1..d5 ratios (should vary across rows):')
print(pred_df[['d1','d2','d3','d4','d5']].head(10).round(4).to_string())


OOS predictions: 252 rows  2015-01-02 → 2015-12-24
d1  ratio mean=1.01031  std=0.00675
d10 ratio mean=1.03884  std=0.04968
d1  pct   mean=1.031%  std=0.675%
signal ON (max ratio >= 1.015): 90.1% of days

Real 2.0× daily log-return std (OOS 2015): 2.833%
Model d1 std (ratio-1):                      0.675%
  ✓ Predictions vary across days — model is working!

Sample d1..d5 ratios (should vary across rows):
                d1      d2      d3      d4      d5
2015-01-02  1.0041  1.0047  0.9844  0.9754  1.0206
2015-01-05  0.9941  0.9833  0.9878  1.0301  1.0485
2015-01-06  1.0117  1.0267  1.0494  1.0637  1.0752
2015-01-07  1.0143  1.0304  1.0480  1.0616  1.0969
2015-01-08  1.0220  1.0523  1.0632  1.0912  1.1060
2015-01-09  1.0204  1.0461  1.0720  1.0896  1.0947
2015-01-12  1.0157  1.0368  1.0538  1.0483  1.0559
2015-01-13  1.0130  1.0370  1.0321  1.0437  1.0356
2015-01-14  1.0114  1.0139  1.0228  1.0182  1.0258
2015-01-15  1.0088  1.0175  1.0097  1.0156  1.0021


## 7. Interactive Chart — cycle_000 OOS + 10-day Input Window Highlight

**Colors:**
- **Yellow** — 10-day input window (the bars the model "sees" before forecasting)
- **★ Lime star** — anchor (today's close, basis for ratios)
- **Green** — predicted 10-day price path (signal: model expects +1.5% hit)
- **Red** — predicted 10-day path (no signal)
- **Blue dashed** — actual realized 10-day path

In [9]:
PRED_STEP = 3   # slider jumps every N days

# Use leveraged series for display — matches the target series used during training
all_close   = df['close_lev']
oos_common  = pred_df.index.intersection(all_close.index)
pred_df_oos = pred_df.loc[oos_common]

n_days = len(TARGET_COLS)

def _build_step(date):
    d_idx  = all_dates.get_loc(date)
    c0     = float(all_close.iloc[d_idx])          # anchor (leveraged close)
    ratios = pred_df_oos.loc[date].values           # d1..d10

    # 10-day input window = the 10 days before anchor
    win_start = max(0, d_idx - WINDOW_SIZE)
    win_x = list(all_dates[win_start : d_idx])
    win_y = [float(all_close.iloc[j]) for j in range(win_start, d_idx)]

    pred_x = [date]; pred_y = [c0]
    act_x  = [date]; act_y  = [c0]
    hover  = [f"<b>Day 0 (anchor)</b> {date.strftime('%Y-%m-%d')}<br>CACT {LEVERAGE_FACTOR}×: {c0:,.1f}"]
    errors = []
    for k in range(n_days):
        fi = d_idx + k + 1
        if fi >= len(all_dates): break
        fd    = all_dates[fi]
        p_eur = c0 * ratios[k]
        a_eur = float(all_close.iloc[fi])
        errors.append(abs(p_eur - a_eur))
        pred_x.append(fd); pred_y.append(p_eur)
        act_x.append(fd);  act_y.append(a_eur)
        hover.append(f"<b>t+{k+1}</b> {fd.strftime('%Y-%m-%d')}<br>"
                     f"Pred: {p_eur:,.1f} ({(ratios[k]-1)*100:+.2f}%)<br>"
                     f"Actual: {a_eur:,.1f}<br>Err: {p_eur-a_eur:+.1f}")

    max_ratio = float(pred_df_oos.loc[date].max())
    signal_on = max_ratio >= ENTRY_THRESHOLD
    mae       = float(np.mean(errors)) if errors else 0.0
    return dict(date=date, c0=c0, win_x=win_x, win_y=win_y,
                pred_x=pred_x, pred_y=pred_y,
                act_x=act_x,  act_y=act_y, hover=hover,
                max_ratio=max_ratio, signal_on=signal_on, mae=mae)

def _title(d):
    sig = '▲ ENTRY SIGNAL' if d['signal_on'] else '— no entry'
    return (f"TKAN v4 cycle_000  |  {d['date'].strftime('%Y-%m-%d')}  anchor={d['c0']:,.1f}  "
            f"|  val_loss={best_val:.6f}  ep={best_ep}/{stopped_at}  lev={LEVERAGE_FACTOR}×  "
            f"|  max pred: {(d['max_ratio']-1)*100:+.2f}%  MAE={d['mae']:.1f}  |  {sig}")

slider_idx = list(range(0, len(oos_common), PRED_STEP))
if slider_idx[-1] != len(oos_common) - 1:
    slider_idx.append(len(oos_common) - 1)

# ── Build figure ──────────────────────────────────────────────────────────
fig = go.Figure()

# Trace 0: full leveraged CACT history (2015 OOS)
oos_hist = all_close.loc[oos_dates[0] : oos_dates[-1]]
fig.add_trace(go.Scatter(
    x=oos_hist.index, y=oos_hist.values,
    mode='lines', name=f'CACT {LEVERAGE_FACTOR}× (2015 OOS)',
    line=dict(color='royalblue', width=1.5)))

# Pre-build first step
first = _build_step(oos_common[slider_idx[0]])

# Trace 1: 10-day input window (yellow)
fig.add_trace(go.Scatter(
    x=first['win_x'], y=first['win_y'],
    mode='lines+markers', name='Input window (10d)',
    line=dict(color='#FFD700', width=3),
    marker=dict(size=6, color='#FFD700', symbol='circle')))

# Trace 2: actual 10-day path
fig.add_trace(go.Scatter(
    x=first['act_x'], y=first['act_y'],
    mode='lines+markers', name='Actual (10d)',
    line=dict(color='royalblue', width=2.5, dash='dash'),
    marker=dict(size=5, color='royalblue')))

# Trace 3: predicted 10-day path
sc0 = '#00e676' if first['signal_on'] else '#ff1744'
fig.add_trace(go.Scatter(
    x=first['pred_x'], y=first['pred_y'],
    mode='lines+markers', name='Predicted (10d)',
    line=dict(color=sc0, width=2.5, dash='dash'),
    marker=dict(size=8, symbol='diamond', color=sc0),
    text=first['hover'], hoverinfo='text'))

# Trace 4: anchor star
fig.add_trace(go.Scatter(
    x=[first['date']], y=[first['c0']],
    mode='markers', name='Anchor',
    marker=dict(size=14, color='lime', symbol='star'),
    text=[first['hover'][0]], hoverinfo='text'))

# ── Slider ────────────────────────────────────────────────────────────────
steps = []
for si in slider_idx:
    d  = _build_step(oos_common[si])
    sc = '#00e676' if d['signal_on'] else '#ff1744'
    steps.append(dict(
        method='update',
        label=d['date'].strftime('%Y-%m-%d'),
        args=[
            {'x': [oos_hist.index, d['win_x'], d['act_x'], d['pred_x'], [d['date']]],
             'y': [oos_hist.values, d['win_y'], d['act_y'], d['pred_y'], [d['c0']]],
             'text': [None, None, None, d['hover'], [d['hover'][0]]],
             'line.color': ['royalblue', '#FFD700', 'royalblue', sc, None],
             'marker.color': ['royalblue', '#FFD700', 'royalblue', sc, 'lime']},
            {'title.text': _title(d)},
            [0, 1, 2, 3, 4],
        ]))

fig.update_layout(
    title=_title(first),
    height=700, width=1400,
    template='plotly_dark',
    hovermode='x unified',
    yaxis_title=f'CACT Total Return  {LEVERAGE_FACTOR}× Leveraged Level',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(b=140),
    annotations=[dict(
        text=f'gold = 10d input window  |  green = entry signal  |  red = no entry  |  blue dashed = actual  |  lev={LEVERAGE_FACTOR}×',
        xref='paper', yref='paper', x=0.5, y=-0.12,
        showarrow=False, font=dict(size=12, color='gray'))],
    sliders=[dict(
        active=0,
        currentvalue=dict(prefix='Forecast from: ', font=dict(size=13), visible=True, xanchor='center'),
        pad=dict(t=55, b=10),
        steps=steps, len=0.95, x=0.025)],
)

fig.show()
